# Extração de protocolos e orientações de saúde da mulher

Este notebook varre as subpastas de `files/` no Google Drive (uma por categoria prioritária), extrai o texto de cada PDF com PyMuPDF e gera um JSON unificado `fontes_saude_mulher_v2.json` com:

- `category` preenchido automaticamente pelo nome da subpasta
- `sensitive: true` para categorias delicadas (violência, saúde mental) — informa o fine-tuning a usar tom mais cuidadoso e encaminhar para serviços especializados
- `name`, `filename`, `content` por documento

**Estrutura esperada de pastas no Drive (nomes exatos, case-sensitive):**

```
AssistenteHospitalar/files/
├── protocolosGinecoObstetricia/   → ginecologia_obstetricia
├── CancerMamaColo/                 → cancer_mama_colo
├── ViolenciaDomestica/             → violencia_domestica   (sensitive)
├── SaudeMental/                    → saude_mental          (sensitive)
└── PlanejamentoFamiliar/           → planejamento_familiar
```

Subpastas não mapeadas em `CATEGORY_MAP` são ignoradas com aviso. PDFs soltos na raiz `files/` também são avisados (não entram no dataset).

Observação: o site do HC-UFMG bloqueia scraping, então os PDFs foram baixados manualmente.

In [ ]:
!pip install -q pymupdf pikepdf

In [19]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FILES_DIR = '/content/drive/MyDrive/AssistenteHospitalar/files'
DRIVE_OUTPUT_JSON = '/content/drive/MyDrive/AssistenteHospitalar/files/fontes_saude_mulher_v2.json'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import json
from pathlib import Path

import fitz

# Mapa: nome EXATO da subpasta no Drive -> metadados da categoria
# Para adicionar uma categoria nova, crie a subpasta em files/ e adicione uma entrada aqui.
CATEGORY_MAP = {
    'protocolosGinecoObstetricia': {'category': 'ginecologia_obstetricia', 'sensitive': False},
    'CancerMamaColo':              {'category': 'cancer_mama_colo',        'sensitive': False},
    'ViolenciaDomestica':          {'category': 'violencia_domestica',     'sensitive': True},
    'SaudeMental':                 {'category': 'saude_mental',            'sensitive': True},
    'PlanejamentoFamiliar':        {'category': 'planejamento_familiar',   'sensitive': False},
}

FILES_DIR = Path(DRIVE_FILES_DIR)
OUTPUT_JSON = Path(DRIVE_OUTPUT_JSON)

print('Diretório de fontes:', FILES_DIR.resolve())
print('Saída JSON:         ', OUTPUT_JSON.resolve())
print()

# Inventário: o que está em cada subpasta + avisos
subdirs = [p for p in FILES_DIR.iterdir() if p.is_dir()]
loose_pdfs = list(FILES_DIR.glob('*.pdf'))

print('Inventário por subpasta:')
total = 0
for subdir in sorted(subdirs):
    pdf_count = len(list(subdir.glob('*.pdf')))
    if subdir.name in CATEGORY_MAP:
        meta = CATEGORY_MAP[subdir.name]
        flag = ' (sensitive)' if meta['sensitive'] else ''
        print(f"  ✓ {subdir.name:<35} -> {meta['category']:<25} {pdf_count} PDFs{flag}")
        total += pdf_count
    else:
        print(f"  ⚠ {subdir.name:<35} -> NÃO MAPEADA, {pdf_count} PDFs serão ignorados")

if loose_pdfs:
    print(f"\n⚠ {len(loose_pdfs)} PDF(s) solto(s) na raiz de files/ (não entram no dataset):")
    for p in loose_pdfs:
        print(f"    - {p.name}")

print(f"\nTotal de PDFs a processar: {total}")

Diretório de fontes: /content/drive/MyDrive/AssistenteHospitalar/files
Saída JSON:          /content/drive/MyDrive/AssistenteHospitalar/files/fontes_saude_mulher_v2.json

Inventário por subpasta:
  ✓ CancerMamaColo                      -> cancer_mama_colo          4 PDFs
  ✓ PlanejamentoFamiliar                -> planejamento_familiar     6 PDFs
  ✓ SaudeMental                         -> saude_mental              2 PDFs (sensitive)
  ✓ ViolenciaDomestica                  -> violencia_domestica       8 PDFs (sensitive)
  ✓ protocolosGinecoObstetricia         -> ginecologia_obstetricia   19 PDFs

Total de PDFs a processar: 39


In [ ]:
def extract_pdf_text(pdf_path):
    try:
        text_parts = []
        with fitz.open(str(pdf_path)) as doc:
            for page in doc:
                text_parts.append(page.get_text())
        return '\n'.join(text_parts).strip()
    except Exception as e:
        print(f'MuPDF error opening {pdf_path.name}: {e}')
        # Try to repair the PDF by rewriting streams with pikepdf and reopen
        try:
            import pikepdf, tempfile
            repaired_path = Path(tempfile.gettempdir()) / (pdf_path.stem + '.repaired.pdf')
            with pikepdf.open(pdf_path) as pdf:
                pdf.save(repaired_path)
            text_parts = []
            with fitz.open(str(repaired_path)) as doc:
                for page in doc:
                    text_parts.append(page.get_text())
            try:
                repaired_path.unlink()
            except Exception:
                pass
            return '\n'.join(text_parts).strip()
        except Exception as e2:
            print(f'Repair with pikepdf failed for {pdf_path.name}: {e2}')
            return ''

def protocol_name_from_filename(pdf_path):
    # Ex.: 'PR_050_Incontinencia_urinaria_mulher_V03.pdf' -> 'PR 050 Incontinencia urinaria mulher V03'
    return pdf_path.stem.replace('_', ' ').strip()

protocol_data = []
counts_by_category = {}

for subdir_name, meta in CATEGORY_MAP.items():
    subdir = FILES_DIR / subdir_name
    if not subdir.exists():
        print(f'  (subpasta ainda não existe: {subdir_name})')
        continue
    for pdf_path in sorted(subdir.glob('*.pdf')):
        text = extract_pdf_text(pdf_path)
        protocol_data.append({
            'category': meta['category'],
            'sensitive': meta['sensitive'],
            'name': protocol_name_from_filename(pdf_path),
            'filename': pdf_path.name,
            'source_folder': subdir_name,
            'content': text,
        })
        counts_by_category[meta['category']] = counts_by_category.get(meta['category'], 0) + 1
        print(f"  [{meta['category']}] {pdf_path.name} ({len(text)} caracteres)")

print(f'\nTotal de documentos extraídos: {len(protocol_data)}')
print('Distribuição por categoria:')
for cat, n in sorted(counts_by_category.items(), key=lambda x: -x[1]):
    print(f'  {cat:<25} {n}')

  [ginecologia_obstetricia] MANUAL-DE-GINECOLOGIA-E-OBSTETRÍCIA.pdf (98847 caracteres)
  [ginecologia_obstetricia] PRT_UMUL_156_Rotura_anteparto_membranas_V03.pdf (16692 caracteres)
  [ginecologia_obstetricia] PRT_UMUL_380_Hemorragia_posparto_V01.pdf (86787 caracteres)
  [ginecologia_obstetricia] PR_050_Incontinencia_urinaria_mulher_V03.pdf (25400 caracteres)
  [ginecologia_obstetricia] PR_128_Atendimento_vitimas_violencia_sexual_domestica_V02.pdf (47748 caracteres)
  [ginecologia_obstetricia] PR_155_Parto_pre_termo_V01.pdf (21062 caracteres)
  [ginecologia_obstetricia] PR_157_Cardiotocografia_GOB_V01.pdf (11669 caracteres)
  [ginecologia_obstetricia] PR_182_Ultrassonografia_Ginecologia_Obstetricia_V01.pdf (20525 caracteres)
  [ginecologia_obstetricia] PR_190_Rotina_gestantes_infectadas_HIV_V01.pdf (19014 caracteres)
  [ginecologia_obstetricia] PR_215_Inducao_tabalho_parto_V01.pdf (18946 caracteres)
  [ginecologia_obstetricia] PR_220_Hipertensao_arterial_cronica_gravidez_V01.pdf (1598

In [17]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON, 'w', encoding='utf-8') as out_file:
    json.dump(protocol_data, out_file, ensure_ascii=False, indent=2)

print('JSON salvo em:', OUTPUT_JSON.resolve())
print('Protocolos processados:', len(protocol_data))

JSON salvo em: /content/drive/MyDrive/AssistenteHospitalar/files/fontes_saude_mulher_v2.json
Protocolos processados: 39
